# Лабораторная работа 12: случайный лес для предсказания возраста ракушки (abalone)

In [3]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score

## 1. Загрузите данные из файла abalone.csv. Это датасет, в котором требуется предсказать возраст ракушки (число колец) по физическим измерениям.

In [4]:
data = pd.read_csv('abalone.csv')
data

,Sex,Length,Diameter,Height,WholeWeight,ShuckedWeight,VisceraWeight,ShellWeight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.1500,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.0700,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.2100,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.1550,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.0550,7
...,...,...,...,...,...,...,...,...,...
4172,F,0.565,0.450,0.165,0.8870,0.3700,0.2390,0.2490,11
4173,M,0.590,0.440,0.135,0.9660,0.4390,0.2145,0.2605,10
4174,M,0.600,0.475,0.205,1.1760,0.5255,0.2875,0.3080,9
4175,F,0.625,0.485,0.150,1.0945,0.5310,0.2610,0.2960,10


## 2. Преобразуйте признак Sex в числовой: значение F должно перейти в -1, I — в 0, M — в 1.

In [5]:
data['Sex'] = data['Sex'].map(lambda x: 1 if x=='M' else (-1 if x=='F' else 0))
data

,Sex,Length,Diameter,Height,WholeWeight,ShuckedWeight,VisceraWeight,ShellWeight,Rings
0,1,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.1500,15
1,1,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.0700,7
2,-1,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.2100,9
3,1,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.1550,10
4,0,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.0550,7
...,...,...,...,...,...,...,...,...,...
4172,-1,0.565,0.450,0.165,0.8870,0.3700,0.2390,0.2490,11
4173,1,0.590,0.440,0.135,0.9660,0.4390,0.2145,0.2605,10
4174,1,0.600,0.475,0.205,1.1760,0.5255,0.2875,0.3080,9
4175,-1,0.625,0.485,0.150,1.0945,0.5310,0.2610,0.2960,10


## 3. Разделите содержимое файлов на признаки и целевую переменную. В последнем столбце записана целевая переменная, в остальных — признаки.

In [6]:
y = data['Rings']
X = data.drop('Rings', axis=1)

## 4. Обучите случайный лес с различным числом деревьев: от 1 до 50 (random_state=1). Для каждого варианта оцените качество на кросс-валидации по 5 блокам (KFold с random_state=1, shuffle=True). В качестве меры качества используйте коэффициент детерминации (R²).

In [7]:
def eval_rf(X, y, mdl):
    kf = KFold(random_state=1, shuffle=True, n_splits=5)
    cv = cross_val_score(mdl, X, y, cv = kf, scoring="r2")
    return cv.mean()

In [8]:
n_tree = np.arange(1, 51)
dict_res = {}
ans = 0
for nt in n_tree:
    rf = RandomForestRegressor(n_estimators=nt, random_state=1)
    dict_res[nt] = eval_rf(X, y, rf)
    if ans == 0 and dict_res[nt] > 0.52:
        ans = nt

## 5. Определите, при каком минимальном количестве деревьев случайный лес показывает качество на кросс-валидации выше 0.52. Это количество и будет ответом на задание.

In [9]:
with open('1.txt', 'w') as file:
    file.write(f'{ans}')

## 6. Обратите внимание на изменение качества по мере роста числа деревьев. Ухудшается ли оно?

Из вывода выше видно, что качество (R²) в целом растёт с увеличением числа деревьев, а после примерно 30 деревьев стабилизируется около 0.53, не ухудшаясь. Таким образом, увеличение числа деревьев не приводит к ухудшению качества.